# GPU Check
Verifica que CUDA y la GPU esten disponibles correctamente.

## 1. Informacion del sistema

In [1]:
import sys
print(f"Python: {sys.version}")

Python: 3.11.0rc1 (main, Aug 12 2022, 10:02:14) [GCC 11.2.0]


In [2]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'nvidia-smi no disponible')

Fri Mar 13 15:58:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.07             Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 1000 Ada Gene...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P4             11W /   35W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. PyTorch + CUDA

In [3]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"CUDA version    : {torch.version.cuda}")
print(f"GPUs detectadas : {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Memoria total : {props.total_memory / 1024**3:.1f} GB")
        print(f"  Compute cap.  : {props.major}.{props.minor}")

PyTorch version : 2.10.0+cu128
CUDA disponible : True
CUDA version    : 12.8
GPUs detectadas : 1

GPU 0: NVIDIA RTX 1000 Ada Generation Laptop GPU
  Memoria total : 6.0 GB
  Compute cap.  : 8.9


## 3. Benchmark: operacion matricial en CPU vs GPU

In [4]:
import torch
import time

size = 4096
a = torch.randn(size, size)
b = torch.randn(size, size)

# CPU
t0 = time.perf_counter()
_ = torch.matmul(a, b)
cpu_ms = (time.perf_counter() - t0) * 1000
print(f"CPU : {cpu_ms:.1f} ms")

# GPU
if torch.cuda.is_available():
    a_gpu = a.cuda()
    b_gpu = b.cuda()
    torch.cuda.synchronize()  # warmup
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()
    gpu_ms = (time.perf_counter() - t0) * 1000

    print(f"GPU : {gpu_ms:.1f} ms")
    print(f"Speedup: {cpu_ms / gpu_ms:.1f}x")
else:
    print('GPU no disponible, benchmark omitido')

CPU : 354.6 ms
GPU : 32.4 ms
Speedup: 11.0x


## 4. Uso de memoria GPU

In [5]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1024**2
    reserved  = torch.cuda.memory_reserved(0)  / 1024**2
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Memoria asignada : {allocated:.1f} MB")
    print(f"Memoria reservada: {reserved:.1f} MB")
    print(f"Memoria total    : {total:.1f} GB")
else:
    print('GPU no disponible')

Memoria asignada : 200.1 MB
Memoria reservada: 276.0 MB
Memoria total    : 6.0 GB
